In [5]:
import http.client
import json
import base64
import os
conn = http.client.HTTPSConnection("poloai.top")

# API_URL = "https://poloai.top/v1beta/models/gemini-2.5-flash-image-preview:generateContent"
# API_KEY = "sk-qeB57n8bPjtYqKxyZ4sFw7MUMy4Uu9bbv0HAsPHH3AFKd8OH"  # 替换为真实 token

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def mime_to_ext(mime: str) -> str:
    return {
        "image/png": ".png",
        "image/jpeg": ".jpg",
        "image/jpg": ".jpg",
        "image/webp": ".webp",
    }.get(mime, ".bin")

def save_gemini_generatecontent_response(raw_bytes: bytes, out_dir: str = "outputs"):
    """
    raw_bytes: res.read() 得到的 bytes（JSON）
    out_dir:   保存目录
    返回：保存的文本路径、图片路径列表
    """
    os.makedirs(out_dir, exist_ok=True)

    # 解析 JSON
    resp = json.loads(raw_bytes.decode("utf-8", errors="replace"))

    texts = []
    image_paths = []

    candidates = resp.get("candidates", [])
    for c_idx, cand in enumerate(candidates):
        content = cand.get("content", {}) or {}
        parts = content.get("parts", []) or []

        for p_idx, part in enumerate(parts):
            # 1) 文本
            if isinstance(part, dict) and "text" in part and part["text"] is not None:
                texts.append(part["text"])

            # 2) 图片（base64）
            inline = None
            if isinstance(part, dict):
                inline = part.get("inlineData") or part.get("inline_data")

            if isinstance(inline, dict) and inline.get("data"):
                mime = inline.get("mimeType") or inline.get("mime_type") or "application/octet-stream"
                ext = mime_to_ext(mime)

                img_bytes = base64.b64decode(inline["data"])
                img_name = f"candidate{c_idx}_part{p_idx}{ext}"
                img_path = os.path.join(out_dir, img_name)

                with open(img_path, "wb") as f:
                    f.write(img_bytes)

                image_paths.append(img_path)
                
    # 保存文本（合并所有 candidates 的 text）
    text_path = os.path.join(out_dir, "response.txt")
    with open(text_path, "w", encoding="utf-8") as f:
        f.write("\n".join(texts).strip() + "\n")

    return text_path, image_paths

base64_image1 = encode_image("cloth.jpg")
base64_image2 = encode_image("model.jpg")

payload = json.dumps({
    "contents": [
        {   
            "role": "user",
            "parts": [
                {"text": "将衣服换上模特"},
                {"inline_data": {"mime_type": "image/jpeg", "data": base64_image1}},
                {"inline_data": {"mime_type": "image/jpeg", "data": base64_image2}},
            ],
        }
    ],
     "generationConfig": {
      "responseModalities": [
         "TEXT",
         "IMAGE"
      ]
   }
})

headers = {
   'Authorization': 'sk-qeB57n8bPjtYqKxyZ4sFw7MUMy4Uu9bbv0HAsPHH3AFKd8OH',
   'Content-Type': 'application/json'
}
conn.request("POST", "/v1beta/models/gemini-2.5-flash-image-preview:generateContent", payload, headers)
res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))
resp = json.loads(data.decode("utf-8", errors="replace"))

text_path, image_paths = save_gemini_generatecontent_response(data, out_dir="gemini_out1")
print("Saved text:", text_path)
print("Saved images:", image_paths)

{"candidates":[{"content":{"role":"model","parts":[{"inlineData":{"mimeType":"image/png","data":"..."}}]},"finishReason":"STOP","index":0,"safetyRatings":null}],"promptFeedback":{"safetyRatings":null},"usageMetadata":{"promptTokenCount":521,"candidatesTokenCount":1290,"totalTokenCount":1811,"promptTokensDetails":[{"modality":"IMAGE","tokenCount":516},{"modality":"TEXT","tokenCount":5}]},"modelVersion":"gemini-2.5-flash-image-preview","responseId":"impXafnjBbGj998Pw9iGkQU"}
Saved text: gemini_out1\response.txt
Saved images: ['gemini_out1\\candidate0_part0.png']
